# Module 1 — Data Pipeline: books.toscrape.com → Zepto-style relational store

This notebook walks through the pipeline end to end and narrates the design decisions.
The heavy lifting lives in the accompanying `.py` modules (`scraper.py`, `clean_transform.py`,
`build_database.py`, `run_queries.py`, `pandas_analysis.py`) so the logic can be unit-tested
and reused; this notebook just orchestrates them and shows the results inline.

**Pipeline stages:** scrape → clean/type-convert → currency-convert → load into normalized SQLite → query with SQL → cross-check with pandas.

## Step 1 — Scrape
We scrape books.toscrape.com category by category (not the generic "all products" pages), because
each category listing page already labels every book with its category — avoiding an extra detail-page
request per book just to discover which category it belongs to. We scrape the first 5 categories fully
(following in-category pagination), which yields well over the required 60 books across well over 3 categories.

In [ ]:
from scraper import scrape_books
import json

raw_rows = scrape_books()
with open('raw_books.json', 'w', encoding='utf-8') as f:
    json.dump(raw_rows, f, indent=2)
print(f'Scraped {len(raw_rows)} raw rows')
raw_rows[:3]

## Step 2 — Clean & type-convert

* `price` (e.g. `"£45.17"`) → `price_gbp` (float): strip the currency symbol and cast.
* `star_rating` (e.g. `"Three"`) → `rating` (int 1–5): direct word→number mapping.
* `availability` (e.g. `"In stock"`) → `in_stock` (bool): substring match on the parsed text.

**Error-handling decision:** if `rating` or `availability` text fails to parse we **drop** the row —
there is no sensible "average" star rating or stock status to impute for a corrupted categorical value.
If `price_gbp` fails to parse we **median-impute** it, using the median price of that book's own category
(falling back to the global median if the category has no valid prices at all) — a missing numeric price is
a reasonable candidate for imputation, unlike a missing categorical label.

In [ ]:
from clean_transform import clean_books, GBP_TO_INR_RATE

df = clean_books(raw_rows)
df.to_csv('cleaned_books.csv', index=False)
print(f'{len(df)} clean rows across {df["category"].nunique()} categories')
print(f'Fixed baseline conversion rate used: 1 GBP = {GBP_TO_INR_RATE} INR (project-defined constant, no date reference)')
df.head()

## Step 3 — Load into a normalized SQLite schema

Two tables, `categories` (PK `category_id`) and `books` (PK `book_id`, FK `category_id` → `categories`).
This is a standard 1-to-many normalization: a category has many books, a book belongs to exactly one category.

In [ ]:
from build_database import build_database, DB_PATH

build_database(df, DB_PATH)

## Step 4 — SQL queries

Six queries, collectively covering `SELECT`/`WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`, `IN`, `BETWEEN`,
and a `JOIN` between `books` and `categories` (top-rated books per category).

In [ ]:
from run_queries import run_all_queries

results = run_all_queries(DB_PATH)

## Step 5 — pandas cross-check: `pd.read_sql` vs `pd.merge`

We read the JOIN query's result back with `pd.read_sql`, and separately reproduce the same result purely
in-memory with `pd.merge` on the two tables (no SQL `JOIN` keyword at all). Both are sorted identically and
compared with `DataFrame.equals` to prove they agree.

In [ ]:
from pandas_analysis import read_sql_examples, merge_equivalent

distinct_categories_sql, join_sql = read_sql_examples(DB_PATH)
merged = merge_equivalent(DB_PATH)

join_sql_sorted = join_sql.reset_index(drop=True)
are_equal = join_sql_sorted.equals(merged)
print('pd.read_sql JOIN output equals pd.merge output:', are_equal)
display(join_sql_sorted.head())
display(merged.head())

## Summary

This module demonstrates the full raw-to-relational data engineering loop: scrape a live public site with
`requests`/`BeautifulSoup`, clean and type-convert messy text fields with explicit, justified error handling,
apply a fixed-rate currency conversion, load into a normalized two-table SQLite schema, query it with SQL
covering all the required clause types plus a join, and cross-validate the join result independently with
pandas — exactly the kind of pipeline a catalog/competitive-intelligence workflow at Zepto would need before
any of this data reaches a dashboard or an analyst's notebook.